# Week 7 — Neural Language Models: Advanced Task

**Student:** Immaculate Mutheu Muli  
**Reg No:** BSSCS/2024/33678  
**Unit:** BIT4133 Natural Language Processing with Deep Learning

**Task:** Build a mini predictive text application that accepts a sentence fragment, predicts the next word, and displays prediction confidence. Bonus features included: a simple text-based interface, a larger dataset (full CBK report), and multiple predictions per query.

## Setup

In [ ]:
!pip install tensorflow pdfplumber --quiet

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
import numpy as np
import re

print('TensorFlow version:', tf.__version__)

## Upload and Process the Full CBK Annual Report (Bonus: Larger Dataset)

In [ ]:
from google.colab import files
import pdfplumber

uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]

text_data = ''
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text_data += page_text + ' '

text_data = text_data.lower()
text_data = re.sub(r'[^a-z\s.]', ' ', text_data)
text_data = re.sub(r'\s+', ' ', text_data).strip()

sentences = re.split(r'(?<=[.])\s+', text_data)
sentences = [s.strip() for s in sentences if 5 <= len(s.split()) <= 20]

print(f'Full report processed.')
print(f'Total usable sentences: {len(sentences)}')

## Build Training Data and Train the Model

Using the full report (Bonus: larger dataset) instead of a small subset, giving the model a richer vocabulary to learn from.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
vocab_size = len(tokenizer.word_index) + 1

input_sequences = []
for sentence in sentences:
    token_list = tokenizer.texts_to_sequences([sentence])[0]
    for i in range(1, len(token_list)):
        input_sequences.append(token_list[:i+1])

max_seq_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_seq_len, padding='pre')

X = input_sequences[:, :-1]
y = to_categorical(input_sequences[:, -1], num_classes=vocab_size)

print(f'Vocabulary size: {vocab_size}')
print(f'Training sequences: {len(input_sequences)}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
model = Sequential([
    Embedding(vocab_size, 64, input_length=max_seq_len - 1),
    LSTM(128),
    Dense(vocab_size, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
history = model.fit(X, y, epochs=60, verbose=1)

print()
print(f'Final training accuracy: {history.history["accuracy"][-1]:.4f}')

## The Mini Predictive Text Application

**Features:**
- Accepts a sentence fragment typed by the user
- Predicts the next word
- Displays prediction confidence
- Bonus: shows multiple top predictions, not just one
- Bonus: simple text-based interactive interface

In [ ]:
def predict_top_words(seed_text, model, tokenizer, max_seq_len, top_n=3):
    """Return the top_n most likely next words with confidence scores."""
    token_list = tokenizer.texts_to_sequences([seed_text])[0]
    token_list = pad_sequences([token_list], maxlen=max_seq_len - 1, padding='pre')
    predicted_probs = model.predict(token_list, verbose=0)[0]

    top_indices = np.argsort(predicted_probs)[-top_n:][::-1]

    index_to_word = {index: word for word, index in tokenizer.word_index.items()}

    results = []
    for idx in top_indices:
        word = index_to_word.get(idx, '<unknown>')
        confidence = predicted_probs[idx]
        results.append((word, confidence))

    return results

print('=' * 65)
print('  MINI PREDICTIVE TEXT APPLICATION — CBK REPORT')
print('=' * 65)
print()
print('  Type a sentence fragment and see the predicted next word.')
print('  Example fragments:')
print('    "the central bank"')
print('    "inflation declined"')
print('    "the monetary policy committee"')
print()
print('  Type "quit" to stop.')
print('=' * 65)

while True:
    print()
    fragment = input('  YOUR SENTENCE FRAGMENT: ').strip()

    if fragment.lower() in ['quit', 'exit', 'stop', '']:
        print()
        print('  Application closed.')
        break

    predictions = predict_top_words(fragment, model, tokenizer, max_seq_len, top_n=3)

    print()
    print(f'  Fragment: "{fragment}"')
    print(f'  Top 3 predicted next words:')
    for rank, (word, confidence) in enumerate(predictions, 1):
        print(f'    {rank}. "{word}"   (confidence: {confidence:.2%})')
    print()
    print('  ── Try another fragment or type "quit" to stop ──')

## Summary

This advanced task built a complete mini predictive text application trained on the **full CBK Annual Report** (bonus: larger dataset). The application:
- Accepts any sentence fragment typed by the user
- Predicts the next word using an Embedding + LSTM + Softmax neural network
- Displays the prediction confidence as a percentage
- Shows the **top 3 predictions** instead of just one (bonus: multiple predictions)
- Runs as an interactive loop so the user can test multiple fragments (bonus: simple interface)